In [ ]:
import pandas as pd, numpy as np
from dask import dataframe as dd
from scipy.stats import norm
from matplotlib import pyplot as plt

In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────────────
# Set these to match config/paths.yaml
import os
COVAR_DIR  = "<set to cfg.gwas.covar_dir in config/paths.yaml>"
MASTER_CSV = "<set to cfg.ukb.master_csv in config/paths.yaml>"
# EXTERNAL inputs (not in config/paths.yaml):
MRI_SAMPLE_LIST = "<EXTERNAL: MRI-cohort sample-inclusion list>"
VISIT_TABLE     = "<EXTERNAL: per-subject imaging-visit table (EID, visit)>"


In [ ]:
T1_cd = os.path.join(COVAR_DIR, "T1_covar_discovery")
T1_cr = os.path.join(COVAR_DIR, "T1_covar_replication")
T2_cd = os.path.join(COVAR_DIR, "T2_covar_discovery")
T2_cr = os.path.join(COVAR_DIR, "T2_covar_replication")

In [ ]:
ukb_all = MASTER_CSV

required covariate:
1. demeaned age
2. demeaned age square
3. demeaned age x demeaned sex
4. demeaned age square x demeaned sex
5. quantile normalized age
6. quantile normalized age square
7. quantile normalized age x demeaned sex
8. 25742 : Mean tfMRI head motion, averaged across space and time points
9. 25742 square
10. quantile normalized 25742
11. 25000: Volumetric scaling from T1 head image to standard space
12. quantile normalized 25000
13. 25756-25759: brain position during scaning stuffs. eg. Scanner lateral (X) brain position 
14. 25756-25759 square
14. quantile normalized 25756-25759
15. quantile normalized 25756-25759 square
16. 53 : Date of attending assessment centre

8, 11 and 13 should undergo outlier removal

In [ ]:
T2_cd

In [ ]:
T1_cd = pd.read_table(T1_cd, sep=' ').drop_duplicates(subset="IID")
T1_cr = pd.read_table(T1_cr, sep=' ').drop_duplicates(subset="IID")
T2_cd = pd.read_table(T2_cd, sep=' ').drop_duplicates(subset="IID")
T2_cr = pd.read_table(T2_cr, sep=' ').drop_duplicates(subset="IID")

In [ ]:
subjects = set(pd.read_table(MRI_SAMPLE_LIST)["ID_1"])

In [ ]:
T1_cd = T1_cd[T1_cd.IID.isin(subjects)]
T1_cr = T1_cr[T1_cr.IID.isin(subjects)]
T2_cd = T2_cd[T2_cd.IID.isin(subjects)]
T2_cr = T2_cr[T2_cr.IID.isin(subjects)]

In [ ]:
combined = pd.concat([T1_cd, T1_cr, T2_cd, T2_cr]).drop_duplicates(subset="IID")
age_mean = combined['AGE'].mean()
sex_mean = combined['SEX'].mean()

In [ ]:
c = pd.read_csv(ukb_all, nrows=0)

In [ ]:
columns = ['eid'] + list(filter(lambda x: x.startswith("25742"), c)) + list(filter(lambda x: x.startswith("25000"), c)) + list(filter(lambda x: x.startswith("2575") and x[4] in ['6', '7', '8', '9'], c)) + list(filter(lambda x: x.startswith("53-") and x[3] in ['2', '3'], c))

In [ ]:
df = dd.read_csv(ukb_all, dtype='object')

In [ ]:
subset = df[df.eid.isin(set(map(str, subjects)))][columns].compute(scheduler='processes')

In [ ]:
visit = pd.read_csv(VISIT_TABLE)[['EID', 'visit']]

In [ ]:
visit = visit.drop_duplicates(subset='EID')
visit_dict = dict(zip(visit.EID, visit.visit))

In [ ]:
v3 = dict(filter(lambda x: x[1]==3, visit_dict.items()))

In [ ]:
assert subset.eid.isin(v3).sum() == 0

In [ ]:
s = subset[~subset.eid.isin(v3)][["eid", "25000-2.0", "25756-2.0", "25757-2.0", "25758-2.0", "25759-2.0", "53-2.0"]].rename(columns={"25000-2.0": "25000", "25756-2.0": "25756", "25757-2.0": "25757", "25758-2.0": "25758", "25759-2.0": "25759", "53-2.0": "53"})

In [ ]:
mask = np.ones(len(s)).astype('bool')
for col in ["25000", "25756", "25757", "25758", "25759"]: #"25742", 
    v = s[col].astype('f')
    m = (v > v.median() - v.std() * 5) & (v < v.median() + v.std() * 5)
    print(len(s) - m.sum())
    mask = mask & m

In [ ]:
s = s[mask].copy()

In [ ]:
d = pd.to_datetime(s["53"])

In [ ]:
s['53'] = (d - d.min()).values/(d.max() - d.min())

In [ ]:
s['53^2'] = s['53']**2

In [ ]:
s['eid'] = s.eid.astype('i')

In [ ]:
def proc_covar(x):
    cols = list(x.columns)
    cols.remove('GE')
    cols.remove('ETH')
    x = x[cols].copy()
    x["AGE^2"] = x["AGE"]**2
    x["SEXxAGE"] = x.SEX * x.AGE
    x["SEXxAGE^2"] = x.SEX * x["AGE^2"]
    x = x.merge(s, left_on="FID", right_on="eid").drop(columns=['eid'])
    return x
    

In [ ]:
T1_cd = proc_covar(T1_cd)
T1_cr = proc_covar(T1_cr)
T2_cd = proc_covar(T2_cd)
T2_cr = proc_covar(T2_cr)

In [ ]:
T1_cd.to_csv("T1_covar_discovery_v2", sep=' ', index=False)
T2_cd.to_csv("T2_covar_discovery_v2", sep=' ', index=False)
T1_cr.to_csv("T1_covar_replication_v2", sep=' ', index=False)
T2_cr.to_csv("T2_covar_replication_v2", sep=' ', index=False)

separate categorical covariates and quantitative covariates

In [ ]:
def sep_covar_qcovar(x):
    clist = ['FID', 'IID', 'SEX', '54']
    qclist = list(x.columns)
    qclist.remove(clist[2])
    qclist.remove(clist[3])
    return x[clist], x[qclist]

In [ ]:
a, b = sep_covar_qcovar(T1_cd)
a.to_csv('T1_ccovar_discovery_v2', sep=' ', index=False)
b.to_csv('T1_qcovar_discovery_v2', sep=' ', index=False)
a, b = sep_covar_qcovar(T1_cr)
a.to_csv('T1_ccovar_replication_v2', sep=' ', index=False)
b.to_csv('T1_qcovar_replication_v2', sep=' ', index=False)
a, b = sep_covar_qcovar(T2_cd)
a.to_csv('T2_ccovar_discovery_v2', sep=' ', index=False)
b.to_csv('T2_qcovar_discovery_v2', sep=' ', index=False)
a, b = sep_covar_qcovar(T2_cr)
a.to_csv('T2_ccovar_replication_v2', sep=' ', index=False)
b.to_csv('T2_qcovar_replication_v2', sep=' ', index=False)